# Day 3: Externalized Prompt Injection & Local SLM Inference
### Learning Objectives & Success Criteria
- Separate prompt engineering from Python code using hot-reloaded YAML templates.
- Run local offline `Qwen2.5-Coder-1.5B` inference with deterministic decoding (`temperature=0.0`).
- Understand static-prefix placement for KV/Prompt Caching.


### Cell 1 — Centralized Settings & Modular Engine Bootstrap


In [ ]:
%load_ext autoreload
%autoreload 2

import os, sys, json, time
sys.path.insert(0, os.path.abspath(".."))
sys.path.insert(0, os.path.abspath("."))

from engine.settings import SETTINGS
from engine.router import ModelRouter
from engine.qdrant_store import QdrantVectorStore
from engine.use_case_loader import UseCaseLoader
from engine.visualizer import StoryVisualizer
from engine.debugger import TraceDebugger
from engine.quiz_helper import render_quiz

# Import PromptRegistry safely regardless of module filename (prompts.py or prompt_registry.py)
try:
    from engine.prompts import PromptRegistry
except ImportError:
    from engine.prompt_registry import PromptRegistry

# All URLs, paths, domain names, and cloud toggles come from SETTINGS (.env + configs/app_config.yaml)
# To test your own domain, see CUSTOM_USE_CASE_GUIDE.md and set ACTIVE_DOMAIN = "your_domain"
ACTIVE_DOMAIN = SETTINGS.active_domain

loader = UseCaseLoader(domain=ACTIVE_DOMAIN)
llm = ModelRouter(local_model_path=SETTINGS.local_model_path, enable_vertex=SETTINGS.enable_vertex_ai)
qdrant = QdrantVectorStore(collection_name=loader.collection_name, path=SETTINGS.qdrant_db_path)
prompts = PromptRegistry()
viz = StoryVisualizer()
dbg = TraceDebugger(debug_mode=False, enable_phoenix=False)

RUNBOOKS = loader.load_runbooks()
INCIDENTS = loader.load_incidents()
qdrant.upsert_documents(RUNBOOKS)

print(f"[Ready] Domain: {loader.domain} | LLM: {llm.provider} | VectorDB: {qdrant.mode} ({loader.collection_name})")


### Cell 2 — Production Incident Selector (0..9)


In [ ]:
# Select any of the 10 production incidents in the active domain (0..9)
INCIDENT_INDEX = 0
ACTIVE_INCIDENT = INCIDENTS[INCIDENT_INDEX]

print(f"Selected Incident: [{ACTIVE_INCIDENT['incident_id']}] {ACTIVE_INCIDENT['title']} (Severity: {ACTIVE_INCIDENT['severity']})")
print("Raw Log:", ACTIVE_INCIDENT["raw_log"])


### Cell 3 — Engineering Concept Brief
**Concept: Externalized YAML Prompts & Deterministic Decoding**
Hot-reloads `configs/prompts/day03_summarizer.yaml` and runs offline `Qwen2.5-Coder-1.5B` without hardcoded prompts or URLs.


### Cell 4 — Traced Execution & X-Ray Visual Storyboard


In [ ]:
dbg.start_run("day03_inference")
with dbg.start_span(f"Day3_Inference_{ACTIVE_INCIDENT['incident_id']}", span_kind="CHAIN") as root_span:
    t0 = time.perf_counter()
    prompt_cfg = prompts.load("day03_summarizer")
    hydrated_user_prompt = prompts.render(prompt_cfg["user_template"], incident_text=ACTIVE_INCIDENT["raw_log"])
    t_prompt = round((time.perf_counter() - t0) * 1000, 2)
    dbg.record_step("load_yaml_prompt", {"template": "day03_summarizer.yaml"}, hydrated_user_prompt, t_prompt, span_kind="CHAIN")

    res = llm.generate(
        prompt=hydrated_user_prompt,
        system_instruction=prompt_cfg["system_prompt"],
        temperature=float(prompt_cfg["temperature"]),
        max_tokens=int(prompt_cfg["max_tokens"]),
    )
    dbg.record_step("qwen_deterministic_gen", hydrated_user_prompt, res["output"], res["latency_ms"], span_kind="LLM")

viz.render_story(
    title=f"Day 3 Externalized Prompt & SLM Inference — {ACTIVE_INCIDENT['incident_id']}",
    stages=[
        {"name": "YAML Prompt Hydrated", "badge": "PROMPT REGISTRY", "latency_ms": t_prompt, "summary": f"temp={prompt_cfg['temperature']}", "payload": hydrated_user_prompt},
        {"name": f"Offline Generation ({res['model']})", "badge": "LOCAL SLM", "latency_ms": res["latency_ms"], "summary": "Zero external URLs or cloud calls", "payload": res["output"]}
    ],
    phoenix_url=dbg.phoenix_url
)


### Cell 5 — Guarded Cloud Parity Check (`SETTINGS.enable_vertex_ai`)


In [ ]:
if SETTINGS.enable_vertex_ai:
    print("Vertex AI Gemini inference parity active.")
else:
    print("ENABLE_VERTEX_AI=false -> Local Qwen inference verified.")


### Cell 6 — Automated Validation Gate


In [ ]:
assert len(res["output"]) > 5
print("[PASS] Day 3 Inference Engineering Gate passed!")


### Cell 7 — Active Lab Experiment
**Hands-On Lab Challenge:** Open `configs/prompts/day03_summarizer.yaml` in VS Code, add a bullet formatting instruction to `system_prompt`, save the file, and re-run Cell 4 **without restarting the kernel** to verify live prompt hot-reloading.


### Cell 8 — Interactive Self-Assessment Questionnaire


In [ ]:
import json, os, sys
from pathlib import Path

repo_root = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

from engine.quiz_helper import render_quiz

with open(repo_root / "configs" / "quizzes" / "day03.json", "r", encoding="utf-8") as f:
    quiz_cfg = json.load(f)

render_quiz(quiz_cfg["title"], quiz_cfg["questions"])


### Cell 9 — Bring Your Own Use Case (BYOUC)
Want to run all 10 notebooks against your own enterprise domain? Follow **[CUSTOM_USE_CASE_GUIDE.md](../CUSTOM_USE_CASE_GUIDE.md)** (`python scripts/scaffold_domain.py --domain your_domain --prefix YOUR-`) and set `ACTIVE_DOMAIN = "your_domain"` in Cell 1.
